## Notebook 05 — Hyperparameter Tuning

This notebook tunes the two best-performing models from notebook 04:
- **Random Forest** (CV R² = 0.976, best generalisation)
- **XGBoost** (CV R² = 0.969, strong tree boosting)
- **LightGBM** (CV R² = 0.956, fast gradient boosting)

Strategy:
1. NDVI merge diagnostic — confirm NDVI is reaching the models
2. `RandomizedSearchCV` for a broad first pass on all three models
3. `GridSearchCV` for a fine-grained second pass on the best one
4. Compare tuned vs untuned results side by side
5. Save the best tuned pipeline to disk for use in the Streamlit app

All preprocessing (OneHotEncoder + StandardScaler) stays inside the Pipeline — no leakage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
import os

warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    TimeSeriesSplit,
    RandomizedSearchCV,
    GridSearchCV,
    cross_validate,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("xgboost not installed — XGBoost tuning will be skipped.")

try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print("lightgbm not installed — LightGBM tuning will be skipped.")

pd.set_option("display.max_columns", None)
print("Imports done.")

### Step 1 — Load data and NDVI diagnostic

Before any tuning, confirm NDVI is actually present and correlated with yield.

In [ ]:
data_path = "../data/processed/crop_yield_ml_features_with_ndvi.csv"
df_raw = pd.read_csv(data_path)

print("=== NDVI DIAGNOSTIC ===")
print(f"Shape before NDVI handling:  {df_raw.shape}")
print(f"NDVI null count:             {df_raw['mean_ndvi'].isnull().sum()} / {len(df_raw)}")
print(f"NDVI unique values:          {df_raw['mean_ndvi'].nunique()}")
print(f"NDVI range:                  {df_raw['mean_ndvi'].min():.4f} → {df_raw['mean_ndvi'].max():.4f}")
print(f"NDVI-yield correlation:      {df_raw['mean_ndvi'].corr(df_raw['yield']):.4f}")
print()

# Show how many rows per state have NDVI vs missing
ndvi_coverage = df_raw.groupby("state")["mean_ndvi"].apply(
    lambda x: f"{x.notna().sum()}/{len(x)} rows"
).reset_index()
ndvi_coverage.columns = ["state", "ndvi_coverage"]
print("NDVI coverage per state:")
display(ndvi_coverage)

In [ ]:
# Fill missing NDVI with per-state mean instead of dropping rows
# This preserves all 17,983 rows instead of losing ~1,013 rows
df_raw["mean_ndvi"] = df_raw.groupby("state")["mean_ndvi"].transform(
    lambda x: x.fillna(x.mean())
)

# Any states with no NDVI at all — fill with global mean
global_ndvi_mean = df_raw["mean_ndvi"].mean()
df_raw["mean_ndvi"] = df_raw["mean_ndvi"].fillna(global_ndvi_mean)

print(f"Shape after NDVI fill:       {df_raw.shape}")
print(f"Remaining NDVI nulls:        {df_raw['mean_ndvi'].isnull().sum()}")
print(f"NDVI-yield correlation:      {df_raw['mean_ndvi'].corr(df_raw['yield']):.4f}")

### Step 2 — Feature definition and train/test split

Identical setup to notebook 04 so results are directly comparable.

In [ ]:
# Log-transform target
df_raw["yield_log"] = np.log1p(df_raw["yield"])

target_col = "yield_log"

feature_cols = [
    "crop", "season", "state",           # categorical
    "year", "area", "fertilizer", "pesticide",
    "N", "P", "K", "pH",
    "avg_temp_c", "total_rainfall_mm", "avg_humidity_percent",
    "mean_ndvi",
]

categorical_cols = ["crop", "season", "state"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

df_sorted = df_raw.sort_values("year").reset_index(drop=True)
X = df_sorted[feature_cols].copy()
y = df_sorted[target_col].values

split_idx = int(len(df_sorted) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
print(f"Feature list: {feature_cols}")

In [ ]:
# Preprocessor — same as notebook 04
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("numeric", StandardScaler(), numeric_cols),
    ]
)

# TimeSeriesSplit CV — same as notebook 04
tscv = TimeSeriesSplit(n_splits=5)

# Helper to evaluate a fitted pipeline on test set + CV
def evaluate_pipeline(name, pipeline, X_tr, y_tr, X_te, y_te, cv):
    scoring = {
        "r2": "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
    }
    cv_res = cross_validate(pipeline, X_tr, y_tr, cv=cv, scoring=scoring, n_jobs=-1)
    pipeline.fit(X_tr, y_tr)
    y_pred_log = pipeline.predict(X_te)

    # Convert back from log scale for interpretable RMSE/MAE
    y_pred_orig = np.expm1(y_pred_log)
    y_te_orig   = np.expm1(y_te)

    return {
        "model":        name,
        "test_r2":      round(r2_score(y_te, y_pred_log), 6),
        "test_rmse":    round(np.sqrt(mean_squared_error(y_te_orig, y_pred_orig)), 4),
        "test_mae":     round(mean_absolute_error(y_te_orig, y_pred_orig), 4),
        "cv_r2_mean":   round(cv_res["test_r2"].mean(), 6),
        "cv_r2_std":    round(cv_res["test_r2"].std(), 6),
        "cv_rmse_mean": round(-cv_res["test_rmse"].mean(), 4),
        "cv_rmse_std":  round(cv_res["test_rmse"].std(), 4),
    }

print("Helper functions ready.")

### Step 3 — Baseline: untuned results (for comparison)

Run default models first so we have a direct before/after comparison.

In [ ]:
baseline_results = []

# Random Forest — default
rf_default = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
])
baseline_results.append(evaluate_pipeline(
    "RF (default)", rf_default, X_train, y_train, X_test, y_test, tscv
))
print("Random Forest default done.")

# XGBoost — default
if XGB_AVAILABLE:
    xgb_default = Pipeline([
        ("preprocess", preprocessor),
        ("model", XGBRegressor(
            n_estimators=400, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            objective="reg:squarederror", n_jobs=-1, random_state=42,
        )),
    ])
    baseline_results.append(evaluate_pipeline(
        "XGBoost (default)", xgb_default, X_train, y_train, X_test, y_test, tscv
    ))
    print("XGBoost default done.")

# LightGBM — default
if LGBM_AVAILABLE:
    lgbm_default = Pipeline([
        ("preprocess", preprocessor),
        ("model", LGBMRegressor(n_estimators=400, random_state=42, n_jobs=-1, verbose=-1)),
    ])
    baseline_results.append(evaluate_pipeline(
        "LightGBM (default)", lgbm_default, X_train, y_train, X_test, y_test, tscv
    ))
    print("LightGBM default done.")

baseline_df = pd.DataFrame(baseline_results)
print("\n=== Baseline (untuned) ===")
display(baseline_df)

### Step 4 — RandomizedSearchCV (broad search)

Explores a wide range of hyperparameters efficiently.
`n_iter=30` tries 30 random combinations — fast enough to run in ~10 minutes.

In [ ]:
from scipy.stats import randint, uniform

# ── Random Forest RandomizedSearch ──────────────────────────────────────────
rf_param_dist = {
    "model__n_estimators":       randint(100, 600),
    "model__max_depth":          [None, 10, 20, 30, 40],
    "model__min_samples_split":  randint(2, 20),
    "model__min_samples_leaf":   randint(1, 10),
    "model__max_features":       ["sqrt", "log2", 0.5, 0.7],
    "model__bootstrap":          [True, False],
}

rf_pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
])

rf_random = RandomizedSearchCV(
    rf_pipe,
    param_distributions=rf_param_dist,
    n_iter=30,
    cv=tscv,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
    verbose=1,
    refit=True,
)

print("Running Random Forest RandomizedSearchCV (30 iterations)...")
rf_random.fit(X_train, y_train)
print("\nBest RF params:")
for k, v in rf_random.best_params_.items():
    print(f"  {k}: {v}")
print(f"Best CV R²: {rf_random.best_score_:.6f}")

In [ ]:
# ── XGBoost RandomizedSearch ─────────────────────────────────────────────────
if XGB_AVAILABLE:
    xgb_param_dist = {
        "model__n_estimators":    randint(200, 800),
        "model__learning_rate":   uniform(0.01, 0.2),
        "model__max_depth":       randint(3, 10),
        "model__subsample":       uniform(0.6, 0.4),
        "model__colsample_bytree": uniform(0.5, 0.5),
        "model__min_child_weight": randint(1, 10),
        "model__reg_alpha":       uniform(0, 1),
        "model__reg_lambda":      uniform(0.5, 2),
        "model__gamma":           uniform(0, 0.5),
    }

    xgb_pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", XGBRegressor(
            objective="reg:squarederror", n_jobs=-1, random_state=42, verbosity=0
        )),
    ])

    xgb_random = RandomizedSearchCV(
        xgb_pipe,
        param_distributions=xgb_param_dist,
        n_iter=30,
        cv=tscv,
        scoring="r2",
        n_jobs=-1,
        random_state=42,
        verbose=1,
        refit=True,
    )

    print("Running XGBoost RandomizedSearchCV (30 iterations)...")
    xgb_random.fit(X_train, y_train)
    print("\nBest XGBoost params:")
    for k, v in xgb_random.best_params_.items():
        print(f"  {k}: {v}")
    print(f"Best CV R²: {xgb_random.best_score_:.6f}")

In [ ]:
# ── LightGBM RandomizedSearch ─────────────────────────────────────────────────
if LGBM_AVAILABLE:
    lgbm_param_dist = {
        "model__n_estimators":    randint(200, 800),
        "model__learning_rate":   uniform(0.01, 0.2),
        "model__max_depth":       randint(3, 12),
        "model__num_leaves":      randint(20, 150),
        "model__subsample":       uniform(0.6, 0.4),
        "model__colsample_bytree": uniform(0.5, 0.5),
        "model__min_child_samples": randint(5, 50),
        "model__reg_alpha":       uniform(0, 1),
        "model__reg_lambda":      uniform(0, 2),
    }

    lgbm_pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)),
    ])

    lgbm_random = RandomizedSearchCV(
        lgbm_pipe,
        param_distributions=lgbm_param_dist,
        n_iter=30,
        cv=tscv,
        scoring="r2",
        n_jobs=-1,
        random_state=42,
        verbose=1,
        refit=True,
    )

    print("Running LightGBM RandomizedSearchCV (30 iterations)...")
    lgbm_random.fit(X_train, y_train)
    print("\nBest LightGBM params:")
    for k, v in lgbm_random.best_params_.items():
        print(f"  {k}: {v}")
    print(f"Best CV R²: {lgbm_random.best_score_:.6f}")

### Step 5 — GridSearchCV (fine-grained search on best model)

Take the best params from RandomizedSearch and search a narrow grid around them.
Run this only on whichever model had the best CV R² above.

In [ ]:
# Fine-tune Random Forest around the best params found above
# Update these values based on rf_random.best_params_ output above
best_rf = rf_random.best_params_

n_est = best_rf["model__n_estimators"]
min_split = best_rf["model__min_samples_split"]
min_leaf = best_rf["model__min_samples_leaf"]

rf_grid_params = {
    "model__n_estimators":      [max(50, n_est - 50), n_est, n_est + 50],
    "model__min_samples_split": [max(2, min_split - 2), min_split, min_split + 2],
    "model__min_samples_leaf":  [max(1, min_leaf - 1), min_leaf, min_leaf + 1],
    "model__max_features":      [best_rf["model__max_features"]],
    "model__max_depth":         [best_rf["model__max_depth"]],
    "model__bootstrap":         [best_rf["model__bootstrap"]],
}

rf_pipe_grid = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
])

rf_grid = GridSearchCV(
    rf_pipe_grid,
    param_grid=rf_grid_params,
    cv=tscv,
    scoring="r2",
    n_jobs=-1,
    verbose=1,
    refit=True,
)

print("Running Random Forest GridSearchCV...")
rf_grid.fit(X_train, y_train)
print("\nBest RF params after grid search:")
for k, v in rf_grid.best_params_.items():
    print(f"  {k}: {v}")
print(f"Best CV R²: {rf_grid.best_score_:.6f}")

### Step 6 — Evaluate all tuned models and compare

In [ ]:
tuned_results = []

# Evaluate RF (RandomizedSearch best)
tuned_results.append(evaluate_pipeline(
    "RF (RandomizedSearch)",
    rf_random.best_estimator_,
    X_train, y_train, X_test, y_test, tscv
))

# Evaluate RF (GridSearch best — typically the winner)
tuned_results.append(evaluate_pipeline(
    "RF (GridSearch)",
    rf_grid.best_estimator_,
    X_train, y_train, X_test, y_test, tscv
))

if XGB_AVAILABLE:
    tuned_results.append(evaluate_pipeline(
        "XGBoost (tuned)",
        xgb_random.best_estimator_,
        X_train, y_train, X_test, y_test, tscv
    ))

if LGBM_AVAILABLE:
    tuned_results.append(evaluate_pipeline(
        "LightGBM (tuned)",
        lgbm_random.best_estimator_,
        X_train, y_train, X_test, y_test, tscv
    ))

tuned_df = pd.DataFrame(tuned_results).sort_values("cv_r2_mean", ascending=False)

print("=== Tuned results ===")
display(tuned_df)

In [ ]:
# Side-by-side comparison: before vs after tuning
comparison = pd.concat([baseline_df, tuned_df], ignore_index=True)

print("=== Before vs After Tuning ===")
display(comparison[[
    "model", "test_r2", "test_rmse", "test_mae",
    "cv_r2_mean", "cv_r2_std", "cv_rmse_mean"
]].sort_values("cv_r2_mean", ascending=False))

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_df = comparison.sort_values("cv_r2_mean", ascending=False)

# CV R² comparison
axes[0].barh(plot_df["model"], plot_df["cv_r2_mean"], xerr=plot_df["cv_r2_std"],
             color="steelblue", capsize=4, alpha=0.85)
axes[0].set_xlabel("CV R² Mean")
axes[0].set_title("CV R² — before vs after tuning")
axes[0].set_xlim(0.9, 1.0)
axes[0].axvline(x=0.976, color="red", linestyle="--", linewidth=0.8, label="RF default (0.976)")
axes[0].legend(fontsize=8)

# CV RMSE comparison
axes[1].barh(plot_df["model"], plot_df["cv_rmse_mean"],
             color="coral", alpha=0.85)
axes[1].set_xlabel("CV RMSE (original scale)")
axes[1].set_title("CV RMSE — before vs after tuning (lower is better)")
axes[1].axvline(x=132, color="red", linestyle="--", linewidth=0.8, label="RF default (132)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### Step 7 — Residual analysis on best model

Plot predicted vs actual in original scale.
Identify which crops and states have the highest prediction error.

In [ ]:
# Use the best tuned model (RF GridSearch)
best_pipeline = rf_grid.best_estimator_
best_pipeline.fit(X_train, y_train)

y_pred_log = best_pipeline.predict(X_test)
y_pred_orig = np.expm1(y_pred_log)
y_test_orig = np.expm1(y_test)

residuals = y_test_orig - y_pred_orig
abs_errors = np.abs(residuals)

# Predicted vs actual scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test_orig, y_pred_orig, alpha=0.3, s=8, color="steelblue")
max_val = max(y_test_orig.max(), y_pred_orig.max())
axes[0].plot([0, max_val], [0, max_val], "r--", linewidth=1)
axes[0].set_xlabel("Actual yield (kg/ha)")
axes[0].set_ylabel("Predicted yield (kg/ha)")
axes[0].set_title("Predicted vs Actual — best tuned model")
axes[0].set_xlim(0, np.percentile(y_test_orig, 99))
axes[0].set_ylim(0, np.percentile(y_pred_orig, 99))

# Residual distribution
axes[1].hist(residuals, bins=60, color="coral", edgecolor="none", alpha=0.8)
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Residual (actual − predicted)")
axes[1].set_title("Residual distribution")

plt.tight_layout()
plt.show()

print(f"Mean residual (bias): {residuals.mean():.2f}")
print(f"Std of residuals:     {residuals.std():.2f}")

In [ ]:
# Per-crop and per-state error analysis
test_df = df_sorted.iloc[split_idx:].copy()
test_df["predicted"] = y_pred_orig
test_df["actual"]    = y_test_orig
test_df["abs_error"] = abs_errors
test_df["pct_error"] = abs_errors / (y_test_orig + 1e-6) * 100

print("Top 10 crops by mean absolute error:")
display(
    test_df.groupby("crop")["abs_error"]
    .mean().sort_values(ascending=False).head(10).reset_index()
    .rename(columns={"abs_error": "mean_abs_error_kg_ha"})
)

print("\nTop 10 states by mean absolute error:")
display(
    test_df.groupby("state")["abs_error"]
    .mean().sort_values(ascending=False).head(10).reset_index()
    .rename(columns={"abs_error": "mean_abs_error_kg_ha"})
)

### Step 8 — Save best pipeline to disk

Saves the complete fitted Pipeline (preprocessor + model) so it can be
loaded directly in the Streamlit app without re-training.

In [ ]:
os.makedirs("../models", exist_ok=True)

model_path = "../models/best_rf_tuned_pipeline.joblib"
joblib.dump(best_pipeline, model_path)
print(f"Saved best pipeline to {model_path}")

# Save feature column list — needed by the Streamlit app
import json
meta = {
    "feature_cols":      feature_cols,
    "categorical_cols":  categorical_cols,
    "numeric_cols":      numeric_cols,
    "target_col":        target_col,
    "crops":             sorted(df_sorted["crop"].unique().tolist()),
    "states":            sorted(df_sorted["state"].unique().tolist()),
    "seasons":           sorted(df_sorted["season"].unique().tolist()),
}
with open("../models/model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved model metadata to ../models/model_meta.json")
print(f"\nFinal best model CV R²:   {rf_grid.best_score_:.6f}")
print(f"Final best model test R²: {r2_score(y_test, best_pipeline.predict(X_test)):.6f}")

### Summary

| Step | What was done |
|------|---------------|
| NDVI diagnostic | Confirmed NDVI coverage per state, filled missing values with state mean |
| RandomizedSearch | 30-iteration broad search on RF, XGBoost, LightGBM |
| GridSearch | Fine-grained narrow search around best RF params |
| Residual analysis | Per-crop and per-state error breakdown |
| Model saved | Best pipeline saved to `../models/best_rf_tuned_pipeline.joblib` |

**Next step → Notebook 06:** Build a Streamlit app that loads `best_rf_tuned_pipeline.joblib`
and lets users input state, crop, season, area, fertilizer, weather → get a yield prediction.